# DB 검색 조회 속도 3차 — 16비트로 고르고 fp32로 다시 정렬 (Colab)

2차(`db_storage_bench_2.ipynb`)에서 벡터를 행 안에 두면 3~5배 빨라졌고, 16비트(halfvec)는 더 빠르고 작지만 순위가 조금 바뀌었습니다(`docs/db/DB_SEARCH_BENCH.md`). 이번에는 **halfvec로 상위 100곳을 고른 뒤 그 100곳만 fp32로 다시 정렬**하면 결과가 fp32와 같은지, 속도는 어떤지 봅니다. 가게를 **법정동 코드 순으로 넣으면** 읽는 페이지가 줄어드는지도 같이 봅니다.

| 이름 | 표 | 쿼리 |
|---|---|---|
| `plain` | fp32 `STORAGE PLAIN`(2차와 같음, 기준) | fp32로 상위 50 |
| `rr` | halfvec `STORAGE PLAIN` + fp32(TOAST) 두 열 | halfvec로 상위 100 → fp32로 상위 50 |
| `rr_sorted` | `rr`과 같고, 법정동 코드 순으로 넣음 | 같음 |
| `plain_sorted` | `plain`과 같고, 법정동 코드 순으로 넣음 | fp32로 상위 50 |

**결과가 fp32와 같은지**: `rr`의 상위 50(가게와 순서)을 `plain`과 질의마다 비교합니다. 같은 fp32로 계산하므로, 상위 50이 halfvec 상위 100 안에 다 들어 있으면 완전히 같아야 합니다.

**Drive에 둘 것**: 1·2차와 같습니다. **결과**: `service_load/bench/db_rerank_<시각>.md`.

- 런타임: GPU(T4). 전체 약 40~50분(설치·복원 8분, 표 네 개 만들기 10~15분, 측정 20~25분).
- 동시 부하는 2차와 같이 클라이언트도 같은 CPU 2개를 쓰므로 표끼리 비교용으로만 봅니다.

In [ ]:
import os, subprocess, multiprocessing
print("CPU", multiprocessing.cpu_count(), "개")
print(subprocess.run("free -g | head -2; df -h /tmp | tail -1; nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%bash
# PostgreSQL 16 + PostGIS 3 + pgvector (PGDG 저장소). 1차와 같은 설정
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y curl ca-certificates lsb-release > /dev/null
install -d /usr/share/postgresql-common/pgdg
curl -s -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-16 postgresql-16-postgis-3 postgresql-16-pgvector > /dev/null
pg_ctlcluster 16 main start || true
runuser -u postgres -- psql -qc "ALTER SYSTEM SET shared_buffers = '2GB'" -c "ALTER SYSTEM SET work_mem = '64MB'" \
  -c "ALTER SYSTEM SET maintenance_work_mem = '1GB'" -c "ALTER USER postgres PASSWORD 'bench'"
pg_ctlcluster 16 main restart
runuser -u postgres -- psql -tAc "select version()"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/store-search-ai")
DUMP = ROOT / "service_load/storesearch_v1.dump"
QUERIES = ROOT / "service_load/bench/service_queries_v1.csv"
MODEL_DIR = ROOT / "runs/finetune/bge_m3_ft_mj_20261014_0743"
OUT_DIR = ROOT / "service_load/bench"
for p in [DUMP, QUERIES, MODEL_DIR]:
    assert p.exists(), f"Drive에 없음: {p}"
print(f"덤프 {DUMP.stat().st_size / 1e9:.2f}GB")

In [ ]:
%%bash
# Drive에서 로컬 디스크로 복사한 뒤 복원
set -e
cp "/content/drive/MyDrive/store-search-ai/service_load/storesearch_v1.dump" /tmp/storesearch_v1.dump
chmod 644 /tmp/storesearch_v1.dump
runuser -u postgres -- dropdb --if-exists storesearch
runuser -u postgres -- createdb storesearch
runuser -u postgres -- pg_restore -d storesearch --no-owner -j 2 /tmp/storesearch_v1.dump 2> /tmp/restore.err || true
echo "복원 경고·오류 $(grep -c ERROR /tmp/restore.err || true)건(public 스키마 등 이미 있는 것은 무시해도 됨)"; grep ERROR /tmp/restore.err | head -5 || true
runuser -u postgres -- psql -d storesearch -tAc "select version_name, status, n_total from data_version; select count(*) from store_snapshot"
rm /tmp/storesearch_v1.dump

In [ ]:
!pip install -q "psycopg[binary,pool]" sentence-transformers==5.7.0 transformers==5.18.0

In [ ]:
# 질의 벡터: 임베딩 서버와 같은 설정(fp32, 128토큰, 정규화)
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

qs = pd.read_csv(QUERIES, dtype=str, keep_default_na=False, encoding="utf-8-sig")
qs = qs[(qs["status"] == "active") & (qs["latitude"] != "") & (qs["longitude"] != "")].reset_index(drop=True)
qs["lat"] = qs["latitude"].astype(float)
qs["lng"] = qs["longitude"].astype(float)
model = SentenceTransformer(str(MODEL_DIR), device="cuda", model_kwargs={"dtype": torch.float32})
model.max_seq_length = 128
qv = model.encode(qs["query"].tolist(), batch_size=128, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
qs["vec"] = ["[" + ",".join(f"{x:.8g}" for x in v) + "]" for v in qv]
del model
torch.cuda.empty_cache()
print(len(qs), "개 질의")

In [ ]:
import re, time, psycopg
DSN = "host=127.0.0.1 dbname=storesearch user=postgres password=bench"
conn = psycopg.connect(DSN, autocommit=True)
conn.execute("create extension if not exists pg_prewarm")
V = conn.execute("select version_id from data_version where status = 'active'").fetchone()[0]
BASE = f"store_snapshot_v{V}"
settings = {k: conn.execute(f"show {k}").fetchone()[0] for k in ["shared_buffers", "work_mem", "max_parallel_workers_per_gather"]}
pg = conn.execute("select extversion from pg_extension where extname = 'vector'").fetchone()[0]
print("버전", V, BASE, settings, "pgvector", pg)

# 질의 위치에서 가장 가까운 가게의 법정동 코드(② 시군구·시도 범위용)
qs["code"] = [conn.execute(
    f"select legal_dong_code from {BASE} where searchable "
    "order by geom <-> ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography limit 1", (r.lng, r.lat)).fetchone()[0]
    for r in qs.itertuples()]

In [ ]:
# 비교용 표 네 개. 기존 파티션을 복사하되 벡터 저장과 넣는 순서만 바꾼다. 색인은 기존 파티션과 같은 정의
cols = [c for (c,) in conn.execute(
    "select column_name from information_schema.columns where table_name = %s order by ordinal_position", (BASE,)).fetchall()]
index_defs = [d for (d,) in conn.execute("select indexdef from pg_indexes where tablename = %s", (BASE,)).fetchall()]


def build(table, with_half, sorted_):
    t = time.perf_counter()
    conn.execute(f"drop table if exists {table}")
    conn.execute(f"create table {table} (like {BASE} including defaults)")
    names = [f'"{c}"' for c in cols]
    sel = [f'"{c}"' for c in cols]
    if with_half:
        # fp32는 TOAST 그대로(다시 정렬할 100곳만 꺼냄), halfvec는 행 안에
        conn.execute(f"alter table {table} add column embedding_half halfvec(1024) not null")
        conn.execute(f"alter table {table} alter column embedding_half set storage plain")
        names.append("embedding_half")
        sel.append("embedding::halfvec(1024)")
    else:
        conn.execute(f"alter table {table} alter column embedding set storage plain")
    order = " order by legal_dong_code, store_id" if sorted_ else ""
    conn.execute(f"insert into {table} ({', '.join(names)}) select {', '.join(sel)} from {BASE}{order}")
    for i, d in enumerate(index_defs):
        conn.execute(re.sub(r"INDEX \S+ ON (ONLY )?\S+", f"INDEX {table}_ix{i} ON public.{table}", d))
    conn.execute(f"vacuum analyze {table}")
    print(table, f"{time.perf_counter() - t:.0f}s")


build("bench_plain", False, False)
build("bench_plain_sorted", False, True)
build("bench_rr", True, False)
build("bench_rr_sorted", True, True)

# (이름, 표, 다시 정렬하나)
VARIANTS = [("plain", "bench_plain", False), ("plain_sorted", "bench_plain_sorted", False),
            ("rr", "bench_rr", True), ("rr_sorted", "bench_rr_sorted", True)]

size_rows = []
for name, table, _ in VARIANTS:
    heap, toast, idx, n = conn.execute(
        "select pg_relation_size(c.oid), coalesce(pg_total_relation_size(nullif(c.reltoastrelid, 0)), 0), "
        "pg_indexes_size(c.oid), c.reltuples from pg_class c where c.relname = %s", (table,)).fetchone()
    size_rows.append(f"| {name} | {heap / 1e9:.2f} | {toast / 1e9:.2f} | {idx / 1e9:.2f} | {n / (heap / 8192):.1f} |")
print("\n".join(size_rows))

In [ ]:
# DB_TABLES.md 10-1절과 같은 범위 조건. 다시 정렬하는 표는 halfvec 상위 100 → fp32 상위 50
RERANK_K = 100
WHERE = "from {T} s where s.version_id = %(ver)s and s.searchable "
EXACT = ("select s.store_id, 1 - (s.embedding <=> %(v)s::vector) as cosine " + WHERE + "{COND}"
         " order by s.embedding <=> %(v)s::vector limit 50")
RERANK = ("select c.store_id, 1 - (c.embedding <=> %(v)s::vector) as cosine from ("
          "select s.store_id, s.embedding " + WHERE + "{COND}"
          f" order by s.embedding_half <=> %(v)s::halfvec limit {RERANK_K}"
          ") c order by c.embedding <=> %(v)s::vector limit 50")
RADIUS = "and ST_DWithin(s.geom, ST_SetSRID(ST_MakePoint(%(lng)s, %(lat)s), 4326)::geography, %(r)s)"
PREFIX = "and s.legal_dong_code like %(p)s || '%%'"
BOX = "and s.geom && ST_MakeEnvelope(%(w)s, %(s)s, %(e)s, %(n)s, 4326)::geography"
CITY, WIDE = (0.03, 0.02), (0.15, 0.10)


def box(r, size):
    return {"w": r.lng - size[0], "s": r.lat - size[1], "e": r.lng + size[0], "n": r.lat + size[1]}


CASES = [
    ("① 반경 1km", RADIUS, lambda r: {"r": 1000}),
    ("① 반경 10km", RADIUS, lambda r: {"r": 10000}),
    ("② 시군구", PREFIX, lambda r: {"p": r.code[:5]}),
    ("② 시도", PREFIX, lambda r: {"p": r.code[:2]}),
    ("③ 지도 도시 화면", BOX, lambda r: box(r, CITY)),
    ("③ 지도 넓은 화면", BOX, lambda r: box(r, WIDE)),
]


def sql_for(cond, table, rerank):
    return (RERANK if rerank else EXACT).replace("{T}", table).replace("{COND}", cond)


def params(r, extra):
    return {"v": r.vec, "ver": V, "lng": r.lng, "lat": r.lat, **extra(r)}


def timed(c, sql, p):
    t = time.perf_counter()
    c.execute(sql, p).fetchall()
    return (time.perf_counter() - t) * 1000


def pct(a, q):
    return float(np.percentile(a, q, method="higher"))


def fmt(x):
    return f"{x:.1f}"


def prewarm(table):
    # 본 표, TOAST 표, 그리고 둘의 색인 전부를 메모리에 올린다
    rels = conn.execute(
        "select c.oid from pg_class c where c.relname = %(t)s "
        "union all select c.reltoastrelid from pg_class c where c.relname = %(t)s and c.reltoastrelid <> 0",
        {"t": table}).fetchall()
    oids = [o for (o,) in rels]
    oids += [i for (i,) in conn.execute("select indexrelid from pg_index where indrelid = any(%s)", (oids,)).fetchall()]
    for oid in oids:
        conn.execute("select pg_prewarm(%s::oid::regclass)", (oid,))

In [ ]:
# 시간: 표마다 메모리에 올린 뒤 질의 전부를 한 번씩
results = {}
for name, table, rerank in VARIANTS:
    prewarm(table)
    for case, cond, extra in CASES:
        q = sql_for(cond, table, rerank)
        ms = [timed(conn, q, params(r, extra)) for r in qs.itertuples()]
        results[(name, case)] = ms
        print(f"{name:12} {case:12} p50 {fmt(pct(ms, 50)):>7} p95 {fmt(pct(ms, 95)):>7}")

In [ ]:
# 결과가 fp32(plain)와 같은지: 경우마다 질의 전부의 상위 50을 가게·순서까지 비교
def tops(table, rerank, cond, extra):
    q = sql_for(cond, table, rerank)
    return [conn.execute(q, params(r, extra)).fetchall() for r in qs.itertuples()]


agree, diffs = [], []
for case, cond, extra in CASES:
    ref = tops("bench_plain", False, cond, extra)
    for name, table, rerank in VARIANTS[1:]:
        got = tops(table, rerank, cond, extra)
        same50 = same10 = top1 = 0
        for qi, (a, b) in enumerate(zip(ref, got)):
            ida, idb = [x[0] for x in a], [x[0] for x in b]
            same50 += ida == idb
            same10 += ida[:10] == idb[:10]
            top1 += ida[:1] == idb[:1]
            if ida != idb and len(diffs) < 10:
                # 어디서 처음 달라지는지와 그 자리의 점수(동점이면 순서만 바뀐 것)
                k = next(i for i in range(min(len(ida), len(idb)) + 1) if i >= min(len(ida), len(idb)) or ida[i] != idb[i])
                sa = f"{a[k][1]:.7f}" if k < len(a) else "-"
                sb = f"{b[k][1]:.7f}" if k < len(b) else "-"
                diffs.append(f"| {name} | {case} | {qs['query'].iloc[qi]} | {k + 1} | {sa} | {sb} | {len(a)} / {len(b)} |")
        n = len(ref)
        agree.append(f"| {case} | {name} | {same50}/{n} | {same10}/{n} | {top1}/{n} |")
print("\n".join(agree))
print("\n".join(diffs))

In [ ]:
# 가장 무거운 경우(같은 쿼리 30번, 첫 번 제외)와 시도 전체 EXPLAIN
v = qs["vec"].iloc[0]
sido, sido_n = conn.execute(f"select left(legal_dong_code, 2), count(*) from {BASE} where searchable "
                            "group by 1 order by 2 desc limit 1").fetchone()
lng, lat, dense_n = conn.execute(
    f"select ST_X(geom::geometry), ST_Y(geom::geometry), (select count(*) from {BASE} t where t.searchable "
    f"and ST_DWithin(t.geom, s.geom, 10000)) as n from {BASE} s tablesample system (1) "
    "where searchable and legal_dong_code like '11%%' order by n desc limit 1").fetchone()
heavy_rows, plans = [], []
for name, table, rerank in VARIANTS:
    prewarm(table)
    cases = [(f"시도 전체(코드 {sido}, {sido_n}곳)", sql_for(PREFIX, table, rerank), {"v": v, "ver": V, "p": sido}),
             (f"서울 가장 빽빽한 곳 10km({dense_n}곳)", sql_for(RADIUS, table, rerank),
              {"v": v, "ver": V, "lng": lng, "lat": lat, "r": 10000})]
    for case, q, p in cases:
        timed(conn, q, p)
        ms = [timed(conn, q, p) for _ in range(30)]
        heavy_rows.append(f"| {name} | {case} | {fmt(pct(ms, 50))} | {fmt(pct(ms, 95))} | {fmt(max(ms))} |")
    plan = conn.execute("explain (analyze, buffers, costs off) " + cases[0][1], cases[0][2]).fetchall()
    plans.append(f"-- {name}\n" + "\n".join(re.sub(r"'\[[^']*\]'", "'[…]'", row[0]) for row in plan))
print("\n".join(heavy_rows))
print(plans[-1])

In [ ]:
# 동시 부하: 표마다 초당 20건 × 60초, 검색 1건 = 1km + 10km(1·2차와 같음). 클라이언트도 같은 VM이라 비교용
from concurrent.futures import ThreadPoolExecutor
from psycopg_pool import ConnectionPool

RATE, SECONDS = 20, 60
rows = list(qs.itertuples())
load_rows = []
for name, table, rerank in VARIANTS:
    prewarm(table)
    q1, q10 = sql_for(RADIUS, table, rerank), sql_for(RADIUS, table, rerank)
    pool = ConnectionPool(DSN, min_size=10, max_size=10, kwargs={"autocommit": True})
    pool.wait()
    lat_ms, errors = [], []

    def one(i):
        r = rows[i % len(rows)]
        t = time.perf_counter()
        try:
            with pool.connection() as c:
                c.execute(q1, params(r, CASES[0][2])).fetchall()
                c.execute(q10, params(r, CASES[1][2])).fetchall()
            lat_ms.append((time.perf_counter() - t) * 1000)
        except Exception as e:
            errors.append(repr(e))

    with ThreadPoolExecutor(32) as ex:
        start = time.perf_counter()
        for i in range(RATE * SECONDS):
            delay = start + i / RATE - time.perf_counter()
            if delay > 0:
                time.sleep(delay)
            ex.submit(one, i)
    pool.close()
    load_rows.append(f"| {name} | {len(lat_ms)} | {len(errors)} | {fmt(pct(lat_ms, 50))} | {fmt(pct(lat_ms, 95))} | "
                     f"{fmt(pct(lat_ms, 99))} | {fmt(max(lat_ms))} |")
    print(load_rows[-1], errors[:2])

In [ ]:
names = [n for n, _, _ in VARIANTS]
report = [f"# DB 검색 조회 속도 3차: halfvec로 고르고 fp32로 다시 정렬 — Colab ({time.strftime('%Y-%m-%d %H:%M')} UTC)\n",
          f"- 환경: Colab CPU {multiprocessing.cpu_count()}개, PostgreSQL 16, pgvector {pg}, shared_buffers {settings['shared_buffers']}, "
          f"work_mem {settings['work_mem']}, max_parallel_workers_per_gather {settings['max_parallel_workers_per_gather']}",
          f"- 데이터 버전 {V}, 질의 {len(qs)}개. 표마다 pg_prewarm 뒤 한 번씩. 시간 = 결과 행을 다 받을 때까지(ms)",
          f"- plain = fp32 STORAGE PLAIN, rr = halfvec PLAIN + fp32 TOAST(halfvec 상위 {RERANK_K} → fp32 상위 50), _sorted = 법정동 코드 순으로 넣음\n",
          "## 크기(GB)\n\n| 표 | 본 행 | TOAST | 색인 | 페이지당 행 |\n|---|---|---|---|---|", *size_rows,
          "\n## p95(ms) — 기준 150\n\n| 경우 | " + " | ".join(names) + " |\n|---|" + "---|" * len(names)]
for case, _, _ in CASES:
    report.append(f"| {case} | " + " | ".join(fmt(pct(results[(n, case)], 95)) for n in names) + " |")
report.append("\n## p50(ms)\n\n| 경우 | " + " | ".join(names) + " |\n|---|" + "---|" * len(names))
for case, _, _ in CASES:
    report.append(f"| {case} | " + " | ".join(fmt(pct(results[(n, case)], 50)) for n in names) + " |")
report += ["\n## plain(fp32)과 결과가 같은 질의 수\n",
           "| 경우 | 표 | 상위 50 가게·순서 모두 같음 | 상위 10 같음 | 1위 같음 |\n|---|---|---|---|---|", *agree,
           "\n### 다른 경우 예시(최대 10개)\n",
           "| 표 | 경우 | 질의 | 처음 다른 순위 | plain 점수 | 표 점수 | 결과 수 |\n|---|---|---|---|---|---|---|", *diffs,
           f"\n## 가장 무거운 경우(질의 '{qs['query'].iloc[0]}', 30번)\n",
           "| 표 | 경우 | p50 | p95 | 최대 |\n|---|---|---|---|---|", *heavy_rows,
           "\n## 시도 전체 EXPLAIN\n\n```\n" + "\n\n".join(plans) + "\n```",
           f"\n## 동시 부하: 초당 {RATE}건 × {SECONDS}초(검색 1건 = 1km + 10km, 클라이언트도 같은 VM)\n",
           "| 표 | 완료 | 오류 | p50 | p95 | p99 | 최대 |\n|---|---|---|---|---|---|---|", *load_rows]
out = OUT_DIR / f"db_rerank_{time.strftime('%Y%m%d_%H%M')}.md"
out.write_text("\n".join(report) + "\n", encoding="utf-8")
print(out)
print("\n".join(report))